In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset, ConcatDataset
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print(f"PyTorch: {torch.__version__}")

IMG_SIZE   = 28
IMG_PIXELS = IMG_SIZE * IMG_SIZE

ModuleNotFoundError: No module named 'torch'

In [ ]:
def get_fashion_mnist(batch_size=128, normalize_11=True):
    mean = (0.5,); std = (0.5,)
    if not normalize_11:
        mean = (0.2860,); std = (0.3530,)
    T = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])
    train_ds = torchvision.datasets.FashionMNIST("./data", train=True,  download=True, transform=T)
    test_ds  = torchvision.datasets.FashionMNIST("./data", train=False, download=True, transform=T)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, drop_last=True)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2)
    return train_dl, test_dl, train_ds.classes

def get_emnist(split="balanced", batch_size=128, normalize_11=True, subset=20000):
    mean = (0.5,); std = (0.5,)
    T = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.squeeze(0).T.unsqueeze(0)),  # orientation fix
        transforms.Normalize(mean, std)
    ])
    train_ds = torchvision.datasets.EMNIST("./data", split=split, train=True,  download=True, transform=T)
    test_ds  = torchvision.datasets.EMNIST("./data", split=split, train=False, download=True, transform=T)
    idx      = list(range(min(subset, len(train_ds))))
    train_dl = DataLoader(Subset(train_ds, idx), batch_size=batch_size, shuffle=True, drop_last=True)
    test_dl  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_dl, test_dl, len(train_ds.classes), train_ds.classes

# Quick test
train_dl, test_dl, classes = get_fashion_mnist()
print(f"FashionMNIST | Train batches: {len(train_dl)} | Classes: {classes}")

In [ ]:
class VAEEncoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(IMG_PIXELS, 512), nn.ReLU(),
            nn.Linear(512, 256),        nn.ReLU(),
        )
        self.fc_mu     = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)

    def forward(self, x):
        h = self.shared(x.view(-1, IMG_PIXELS))
        return self.fc_mu(h), self.fc_logvar(h)


class VAEDecoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, 512),        nn.ReLU(),
            nn.Linear(512, IMG_PIXELS), nn.Sigmoid(),
        )

    def forward(self, z):
        return self.net(z).view(-1, 1, IMG_SIZE, IMG_SIZE)


class VAE(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.encoder    = VAEEncoder(latent_dim)
        self.decoder    = VAEDecoder(latent_dim)
        self.latent_dim = latent_dim

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

    def encode(self, x): return self.encoder(x)
    def decode(self, z): return self.decoder(z)


def vae_loss(recon, x, mu, logvar, kl_weight=1.0):
    recon_loss = F.binary_cross_entropy(recon.view(-1, IMG_PIXELS),
                                         x.view(-1, IMG_PIXELS), reduction="sum") / x.size(0)
    kl_loss    = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return recon_loss + kl_weight * kl_loss, recon_loss.item(), kl_loss.item()

print("VAE architecture defined.")

In [ ]:
def train_vae(latent_dim=32, epochs=20, batch_size=128, dataset="fashion"):
    if dataset == "fashion":
        dl, test_dl, _ = get_fashion_mnist(batch_size, normalize_11=False)
    else:
        dl, test_dl, _, _ = get_emnist(batch_size=batch_size, normalize_11=False)

    model = VAE(latent_dim).to(device)
    opt   = optim.Adam(model.parameters(), lr=1e-3)
    history = {"loss": [], "recon": [], "kl": []}

    for epoch in range(1, epochs + 1):
        model.train()
        kl_w = min(1.0, epoch / 10)           # KL annealing: warm up over 10 epochs
        ep_loss = ep_recon = ep_kl = 0.0
        for x, _ in dl:
            x = x.to(device)
            recon, mu, logvar = model(x)
            loss, r, k = vae_loss(recon, x, mu, logvar, kl_weight=kl_w)
            opt.zero_grad(); loss.backward(); opt.step()
            ep_loss += loss.item(); ep_recon += r; ep_kl += k
        n = len(dl)
        history["loss"].append(ep_loss / n)
        history["recon"].append(ep_recon / n)
        history["kl"].append(ep_kl / n)
        print(f"[VAE z={latent_dim}] Epoch {epoch:02d}/{epochs} | "
              f"Loss {ep_loss/n:.2f} | Recon {ep_recon/n:.2f} | KL {ep_kl/n:.2f} | KLw {kl_w:.2f}")

    return model, history, dl, test_dl

vae, vae_history, vae_dl, vae_test_dl = train_vae(latent_dim=32, epochs=20)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(vae_history["loss"],  label="Total"); axes[0].plot(vae_history["recon"], label="Recon (BCE)")
axes[0].set_title("VAE Loss"); axes[0].legend()
axes[1].plot(vae_history["kl"], color="orange"); axes[1].set_title("KL Divergence")
plt.suptitle("VAE (z=32) Training Curves", fontweight="bold"); plt.tight_layout(); plt.show()

In [ ]:
x_batch, _ = next(iter(vae_test_dl))
x_in = x_batch[:8].to(device)
with torch.no_grad():
    recon, _, _ = vae(x_in)

grid = vutils.make_grid(torch.cat([x_in.cpu(), recon.cpu()]), nrow=8, normalize=True)
plt.figure(figsize=(10, 2.5))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.axis("off"); plt.title("Top: Original  |  Bottom: Reconstructed"); plt.show()

In [ ]:
vae.eval()
with torch.no_grad():
    z = torch.randn(64, vae.latent_dim, device=device)
    samples = vae.decode(z)

grid = vutils.make_grid(samples, nrow=8, normalize=True)
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.axis("off"); plt.title("VAE Random Samples  z ~ N(0,I)"); plt.show()

In [ ]:
x1 = x_batch[0].to(device); x2 = x_batch[1].to(device)
steps = 10
with torch.no_grad():
    mu1, _ = vae.encode(x1.unsqueeze(0))
    mu2, _ = vae.encode(x2.unsqueeze(0))
    alphas  = torch.linspace(0, 1, steps, device=device)
    interp  = torch.stack([(1 - a) * mu1 + a * mu2 for a in alphas]).squeeze(1)
    imgs    = vae.decode(interp)

grid = vutils.make_grid(imgs, nrow=steps, normalize=True)
plt.figure(figsize=(steps * 1.2, 1.8))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.axis("off"); plt.title("Latent Space Traversal  (Image A → Image B)"); plt.show()

In [ ]:
results_zdim = {}
for z_dim in [8, 32, 128]:
    v, h, _, _ = train_vae(latent_dim=z_dim, epochs=10)
    results_zdim[z_dim] = {"recon": h["recon"][-1], "kl": h["kl"][-1]}
    print(f"z={z_dim:3d} | Recon={h['recon'][-1]:.4f} | KL={h['kl'][-1]:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([str(k) for k in results_zdim], [v["recon"] for v in results_zdim.values()], color=["#4e79a7","#f28e2b","#e15759"])
ax.set_xlabel("Latent Dimension"); ax.set_ylabel("Reconstruction Loss")
ax.set_title("Reconstruction Loss vs Latent Dimension"); plt.tight_layout(); plt.show()

In [ ]:
class GANGenerator(nn.Module):
    def __init__(self, noise_dim=100):
        super().__init__()
        self.noise_dim = noise_dim
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 256),   nn.LeakyReLU(0.2), nn.BatchNorm1d(256),
            nn.Linear(256, 512),          nn.LeakyReLU(0.2), nn.BatchNorm1d(512),
            nn.Linear(512, 1024),         nn.LeakyReLU(0.2), nn.BatchNorm1d(1024),
            nn.Linear(1024, IMG_PIXELS),  nn.Tanh(),
        )
    def forward(self, z):
        return self.net(z).view(-1, 1, IMG_SIZE, IMG_SIZE)


class GANDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(IMG_PIXELS, 512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(512, 256),         nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(256, 1),           nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x.view(-1, IMG_PIXELS))

print("GAN (FC) architecture defined.")

In [ ]:
def train_gan(epochs=50, noise_dim=100, batch_size=128, dataset="fashion"):
    dl, _, _ = get_fashion_mnist(batch_size) if dataset == "fashion" \
               else get_emnist(batch_size=batch_size)[:3]

    G = GANGenerator(noise_dim).to(device)
    D = GANDiscriminator().to(device)
    opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
    criterion = nn.BCELoss()
    fixed_z = torch.randn(64, noise_dim, device=device)
    history = {"G": [], "D": []}

    for epoch in range(1, epochs + 1):
        G.train(); D.train()
        ep_G = ep_D = 0.0
        for x_real, _ in dl:
            B = x_real.size(0); x_real = x_real.to(device)
            real_lb = torch.full((B, 1), 0.9, device=device)   # label smoothing
            fake_lb = torch.zeros(B, 1, device=device)

            # --- Train D ---
            z = torch.randn(B, noise_dim, device=device)
            loss_D = criterion(D(x_real), real_lb) + criterion(D(G(z).detach()), fake_lb)
            opt_D.zero_grad(); loss_D.backward(); opt_D.step()

            # --- Train G (non-saturating) ---
            z = torch.randn(B, noise_dim, device=device)
            loss_G = criterion(D(G(z)), torch.ones(B, 1, device=device))
            opt_G.zero_grad(); loss_G.backward(); opt_G.step()

            ep_G += loss_G.item(); ep_D += loss_D.item()

        n = len(dl)
        history["G"].append(ep_G / n); history["D"].append(ep_D / n)
        print(f"[GAN] Epoch {epoch:02d}/{epochs} | G {ep_G/n:.4f} | D {ep_D/n:.4f}")

        if epoch % 10 == 0:
            G.eval()
            with torch.no_grad():
                imgs = G(fixed_z)
            grid = vutils.make_grid(imgs, nrow=8, normalize=True, value_range=(-1,1))
            plt.figure(figsize=(8,8)); plt.imshow(grid.cpu().permute(1,2,0).squeeze(), cmap="gray")
            plt.axis("off"); plt.title(f"GAN Samples — Epoch {epoch}"); plt.show()

    return G, D, history

gan_G, gan_D, gan_history = train_gan(epochs=50)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(gan_history["G"], label="Generator Loss")
ax.plot(gan_history["D"], label="Discriminator Loss")
ax.set_xlabel("Epoch"); ax.set_title("Vanilla GAN — Training Loss Curves"); ax.legend()
plt.tight_layout(); plt.show()

# Diversity check: std of pixel values across 500 generated samples
gan_G.eval()
with torch.no_grad():
    z = torch.randn(500, 100, device=device)
    fake = gan_G(z).cpu()
pixel_std = fake.std(dim=0).mean().item()
print(f"Pixel-space diversity (std): {pixel_std:.4f}  (higher = more diverse, lower = mode collapse)")

In [ ]:
class CGANGenerator(nn.Module):
    def __init__(self, noise_dim=100, num_classes=10, emb_dim=50):
        super().__init__()
        self.emb = nn.Embedding(num_classes, emb_dim)
        self.net = nn.Sequential(
            nn.Linear(noise_dim + emb_dim, 256),  nn.LeakyReLU(0.2), nn.BatchNorm1d(256),
            nn.Linear(256, 512),                   nn.LeakyReLU(0.2), nn.BatchNorm1d(512),
            nn.Linear(512, 1024),                  nn.LeakyReLU(0.2), nn.BatchNorm1d(1024),
            nn.Linear(1024, IMG_PIXELS),            nn.Tanh(),
        )
        self.noise_dim = noise_dim; self.num_classes = num_classes
    def forward(self, z, y):
        return self.net(torch.cat([z, self.emb(y)], dim=1)).view(-1, 1, IMG_SIZE, IMG_SIZE)


class CGANDiscriminator(nn.Module):
    def __init__(self, num_classes=10, emb_dim=50):
        super().__init__()
        self.emb = nn.Embedding(num_classes, emb_dim)
        self.net = nn.Sequential(
            nn.Linear(IMG_PIXELS + emb_dim, 512),  nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(512, 256),                    nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(256, 1),                      nn.Sigmoid(),
        )
    def forward(self, x, y):
        return self.net(torch.cat([x.view(-1, IMG_PIXELS), self.emb(y)], dim=1))

print("CGAN architecture defined.")

In [ ]:
def train_cgan(epochs=50, noise_dim=100, num_classes=10, batch_size=128, dataset="fashion"):
    if dataset == "fashion":
        dl, _, cls = get_fashion_mnist(batch_size)
        num_classes = len(cls)
    else:
        dl, _, num_classes, _ = get_emnist(batch_size=batch_size)

    G = CGANGenerator(noise_dim, num_classes).to(device)
    D = CGANDiscriminator(num_classes).to(device)
    opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
    criterion = nn.BCELoss()
    fixed_z = torch.randn(num_classes * 5, noise_dim, device=device)
    fixed_y = torch.arange(num_classes, device=device).repeat_interleave(5)
    history = {"G": [], "D": []}

    for epoch in range(1, epochs + 1):
        G.train(); D.train()
        ep_G = ep_D = 0.0
        for x_real, y_real in dl:
            B = x_real.size(0)
            x_real = x_real.to(device); y_real = y_real.to(device)

            z = torch.randn(B, noise_dim, device=device)
            y_fake = torch.randint(0, num_classes, (B,), device=device)
            loss_D = criterion(D(x_real, y_real), torch.full((B,1),0.9,device=device)) + \
                     criterion(D(G(z,y_fake).detach(), y_fake), torch.zeros(B,1,device=device))
            opt_D.zero_grad(); loss_D.backward(); opt_D.step()

            z = torch.randn(B, noise_dim, device=device)
            y_fake = torch.randint(0, num_classes, (B,), device=device)
            loss_G = criterion(D(G(z, y_fake), y_fake), torch.ones(B,1,device=device))
            opt_G.zero_grad(); loss_G.backward(); opt_G.step()

            ep_G += loss_G.item(); ep_D += loss_D.item()

        n = len(dl)
        history["G"].append(ep_G/n); history["D"].append(ep_D/n)
        print(f"[CGAN] Epoch {epoch:02d}/{epochs} | G {ep_G/n:.4f} | D {ep_D/n:.4f}")

        if epoch % 10 == 0:
            G.eval()
            with torch.no_grad(): imgs = G(fixed_z, fixed_y)
            grid = vutils.make_grid(imgs, nrow=num_classes, normalize=True, value_range=(-1,1))
            plt.figure(figsize=(num_classes*0.8, 5))
            plt.imshow(grid.cpu().permute(1,2,0).squeeze(), cmap="gray")
            plt.axis("off"); plt.title(f"CGAN — Epoch {epoch} (each row = one class)"); plt.show()

    return G, D, history, num_classes

cgan_G, cgan_D, cgan_history, N_CLS = train_cgan(epochs=50)

In [ ]:
cgan_G.eval()
z_fixed = torch.randn(1, 100, device=device).repeat(N_CLS, 1)
y_all   = torch.arange(N_CLS, device=device)
with torch.no_grad():
    imgs = cgan_G(z_fixed, y_all)
grid = vutils.make_grid(imgs, nrow=N_CLS, normalize=True, value_range=(-1,1))
plt.figure(figsize=(N_CLS * 0.9, 1.8))
plt.imshow(grid.cpu().permute(1,2,0).squeeze(), cmap="gray")
plt.axis("off"); plt.title("Label Interpolation — Fixed z, Varying class label y"); plt.show()

In [ ]:
cgan_G.eval(); cgan_D.eval()
correct = total = 0
with torch.no_grad():
    for x_real, y_real in test_dl:
        x_real = x_real.to(device); y_real = y_real.to(device)
        wrong_y = (y_real + 1) % N_CLS              # shift labels by 1
        score_correct = cgan_D(x_real, y_real)
        score_wrong   = cgan_D(x_real, wrong_y)
        correct += (score_correct > score_wrong).sum().item()
        total   += y_real.size(0)
print(f"Mismatched label attack — D prefers correct label: {correct/total:.4f}  (ideal: ~1.0)")

In [ ]:
def _dcgan_init(m):
    cn = m.__class__.__name__
    if "Conv" in cn:
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif "BatchNorm" in cn:
        nn.init.normal_(m.weight, 1.0, 0.02); nn.init.constant_(m.bias, 0.0)


class DCGANGenerator(nn.Module):
    def __init__(self, noise_dim=100):
        super().__init__()
        self.noise_dim = noise_dim
        self.project = nn.Linear(noise_dim, 128 * 7 * 7)
        self.net = nn.Sequential(
            # 7×7 → 14×14
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(True),
            # 14×14 → 28×28
            nn.ConvTranspose2d(64,  1, 4, stride=2, padding=1, bias=False),
            nn.Tanh(),
        )
    def forward(self, z):
        return self.net(self.project(z).view(-1, 128, 7, 7))


class DCGANDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,  64, 4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
        )
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(128*7*7, 1), nn.Sigmoid())
    def forward(self, x):
        return self.fc(self.net(x))

print("DCGAN architecture defined.")

In [ ]:
def train_dcgan(epochs=50, noise_dim=100, batch_size=128, use_spectral_norm=False, dataset="fashion"):
    dl, _, _ = get_fashion_mnist(batch_size) if dataset == "fashion" \
               else get_emnist(batch_size=batch_size)[:3]

    G = DCGANGenerator(noise_dim).to(device)
    D = DCGANDiscriminator().to(device)
    G.apply(_dcgan_init); D.apply(_dcgan_init)

    if use_spectral_norm:
        for m in D.modules():
            if isinstance(m, nn.Conv2d):
                nn.utils.spectral_norm(m)

    opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
    criterion = nn.BCELoss()
    fixed_z = torch.randn(64, noise_dim, device=device)
    history = {"G": [], "D": []}

    for epoch in range(1, epochs + 1):
        G.train(); D.train()
        ep_G = ep_D = 0.0
        for x_real, _ in dl:
            B = x_real.size(0); x_real = x_real.to(device)
            real_lb = torch.full((B,1), 0.9, device=device)
            fake_lb = torch.zeros(B,1, device=device)

            z = torch.randn(B, noise_dim, device=device)
            loss_D = criterion(D(x_real), real_lb) + criterion(D(G(z).detach()), fake_lb)
            opt_D.zero_grad(); loss_D.backward(); opt_D.step()

            z = torch.randn(B, noise_dim, device=device)
            loss_G = criterion(D(G(z)), torch.ones(B,1,device=device))
            opt_G.zero_grad(); loss_G.backward(); opt_G.step()

            ep_G += loss_G.item(); ep_D += loss_D.item()

        n = len(dl)
        history["G"].append(ep_G/n); history["D"].append(ep_D/n)
        print(f"[DCGAN{'(SN)' if use_spectral_norm else ''}] Epoch {epoch:02d}/{epochs} | G {ep_G/n:.4f} | D {ep_D/n:.4f}")

        if epoch % 10 == 0:
            G.eval()
            with torch.no_grad(): imgs = G(fixed_z)
            grid = vutils.make_grid(imgs, nrow=8, normalize=True, value_range=(-1,1))
            plt.figure(figsize=(8,8)); plt.imshow(grid.cpu().permute(1,2,0).squeeze(), cmap="gray")
            plt.axis("off"); plt.title(f"DCGAN — Epoch {epoch}"); plt.show()

    return G, D, history

dcgan_G, dcgan_D, dcgan_history = train_dcgan(epochs=50)

In [ ]:
dcgan_sn_G, dcgan_sn_D, dcgan_sn_history = train_dcgan(epochs=50, use_spectral_norm=True)

In [ ]:
# Train a quick EMNIST DCGAN first, then do: encode("a") + encode("B") - encode("b")
# Here we demonstrate with random z arithmetic as a proxy
dcgan_G.eval()
with torch.no_grad():
    za = torch.randn(1, 100, device=device)
    zB = torch.randn(1, 100, device=device)
    zb = torch.randn(1, 100, device=device)
    z_combo = za + zB - zb
    imgs = torch.cat([dcgan_G(za), dcgan_G(zB), dcgan_G(zb), dcgan_G(z_combo)])

grid = vutils.make_grid(imgs, nrow=4, normalize=True, value_range=(-1,1))
plt.figure(figsize=(6, 2))
plt.imshow(grid.cpu().permute(1,2,0).squeeze(), cmap="gray")
plt.axis("off"); plt.title('Latent Arithmetic: z_a + z_B − z_b  (left→right)'); plt.show()

In [ ]:
all_histories = {
    "Vanilla GAN": gan_history,
    "CGAN":        cgan_history,
    "DCGAN":       dcgan_history,
    "DCGAN+SN":    dcgan_sn_history,
}

fig, axes = plt.subplots(1, len(all_histories), figsize=(5 * len(all_histories), 4))
for ax, (name, h) in zip(axes, all_histories.items()):
    ax.plot(h["G"], label="G Loss")
    ax.plot(h["D"], label="D Loss")
    ax.set_title(name); ax.set_xlabel("Epoch"); ax.legend(fontsize=8)
plt.suptitle("Generator vs Discriminator Loss — All Models", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
models = {"VAE": vae, "GAN-G": gan_G, "GAN-D": gan_D,
          "CGAN-G": cgan_G, "DCGAN-G": dcgan_G, "DCGAN-D": dcgan_D}

names = list(models.keys())
params = [sum(p.numel() for p in m.parameters()) / 1e6 for m in models.values()]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(names, params, color=plt.cm.Set2(np.linspace(0,1,len(names))))
ax.bar_label(bars, fmt="%.2fM", fontsize=9)
ax.set_ylabel("Parameters (M)"); ax.set_title("Parameter Count Comparison")
plt.tight_layout(); plt.show()

In [ ]:
class LeNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.clf = nn.Sequential(nn.Flatten(), nn.Linear(64*7*7, 128), nn.ReLU(), nn.Linear(128, num_classes))
    def forward(self, x): return self.clf(self.features(x))


def train_eval_clf(train_loader, test_loader, epochs=10, label=""):
    clf = LeNet(num_classes=10).to(device)
    opt = optim.Adam(clf.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    for ep in range(epochs):
        clf.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = crit(clf(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
    clf.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            correct += (clf(x).argmax(1) == y).sum().item(); total += y.size(0)
    acc = correct / total
    print(f"[{label}] Test Accuracy: {acc:.4f}"); return acc

T_01 = transforms.Compose([transforms.ToTensor()])
full_train = torchvision.datasets.FashionMNIST("./data", train=True,  download=True, transform=T_01)
full_test  = torchvision.datasets.FashionMNIST("./data", train=False, download=True, transform=T_01)

n_real   = int(0.1 * len(full_train))
real_dl  = DataLoader(Subset(full_train, list(range(n_real))), batch_size=128, shuffle=True)
test_dl2 = DataLoader(full_test, batch_size=128, shuffle=False)

base_acc = train_eval_clf(real_dl, test_dl2, epochs=10, label="Baseline 10% Real")

In [ ]:
cgan_G.eval()
syn_x, syn_y = [], []
with torch.no_grad():
    for cls in range(N_CLS):
        z = torch.randn(600, 100, device=device)
        y = torch.full((600,), cls, dtype=torch.long, device=device)
        imgs = (cgan_G(z, y).cpu() + 1) / 2          # [-1,1] → [0,1]
        syn_x.append(imgs); syn_y.append(torch.full((600,), cls))

syn_ds   = TensorDataset(torch.cat(syn_x), torch.cat(syn_y))
combo_ds = ConcatDataset([Subset(full_train, list(range(n_real))), syn_ds])
aug_dl   = DataLoader(combo_ds, batch_size=128, shuffle=True)

aug_acc = train_eval_clf(aug_dl, test_dl2, epochs=10, label="Augmented 50% Real + 50% Synthetic")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Baseline\n(10% real)", "Augmented\n(+CGAN synthetic)"],
       [base_acc, aug_acc], color=["#4e79a7","#f28e2b"], width=0.5)
ax.set_ylim(0, 1); ax.set_ylabel("Test Accuracy")
ax.set_title(f"Data Augmentation — Δ = {aug_acc - base_acc:+.4f}")
for i, v in enumerate([base_acc, aug_acc]):
    ax.text(i, v + 0.01, f"{v:.4f}", ha="center", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
def gradient_penalty(D, real, fake):
    B = real.size(0)
    alpha = torch.rand(B, 1, 1, 1, device=device)
    interp = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    d_out  = D(interp)
    grads  = torch.autograd.grad(d_out, interp,
                                  grad_outputs=torch.ones_like(d_out),
                                  create_graph=True, retain_graph=True)[0]
    return ((grads.norm(2, dim=[1,2,3]) - 1) ** 2).mean()


def train_wgan_gp(epochs=30, noise_dim=100, lam=10, batch_size=128):
    dl, _, _ = get_fashion_mnist(batch_size)
    G = DCGANGenerator(noise_dim).to(device); G.apply(_dcgan_init)
    D = DCGANDiscriminator().to(device);       D.apply(_dcgan_init)
    D.fc[-1] = nn.Identity()                  # remove Sigmoid for Wasserstein

    opt_G = optim.Adam(G.parameters(), lr=1e-4, betas=(0.0, 0.9))
    opt_D = optim.Adam(D.parameters(), lr=1e-4, betas=(0.0, 0.9))
    fixed_z = torch.randn(64, noise_dim, device=device)
    history = {"G": [], "D": []}

    for epoch in range(1, epochs + 1):
        G.train(); D.train()
        ep_G = ep_D = 0.0
        for x_real, _ in dl:
            B = x_real.size(0); x_real = x_real.to(device)

            # Train critic 5×
            for _ in range(5):
                z = torch.randn(B, noise_dim, device=device)
                x_fake = G(z).detach()
                gp = gradient_penalty(D, x_real, x_fake)
                loss_D = -D(x_real).mean() + D(x_fake).mean() + lam * gp
                opt_D.zero_grad(); loss_D.backward(); opt_D.step()

            z = torch.randn(B, noise_dim, device=device)
            loss_G = -D(G(z)).mean()
            opt_G.zero_grad(); loss_G.backward(); opt_G.step()

            ep_G += loss_G.item(); ep_D += loss_D.item()

        n = len(dl)
        history["G"].append(ep_G/n); history["D"].append(ep_D/n)
        print(f"[WGAN-GP] Epoch {epoch:02d}/{epochs} | G {ep_G/n:.4f} | D {ep_D/n:.4f}")

        if epoch % 10 == 0:
            G.eval()
            with torch.no_grad(): imgs = G(fixed_z)
            grid = vutils.make_grid(imgs, nrow=8, normalize=True, value_range=(-1,1))
            plt.figure(figsize=(8,8)); plt.imshow(grid.cpu().permute(1,2,0).squeeze(), cmap="gray")
            plt.axis("off"); plt.title(f"WGAN-GP — Epoch {epoch}"); plt.show()

    return G, D, history

wgan_G, wgan_D, wgan_history = train_wgan_gp(epochs=30)

In [ ]:
def train_lsgan(epochs=30, noise_dim=100, batch_size=128):
    dl, _, _ = get_fashion_mnist(batch_size)
    G = DCGANGenerator(noise_dim).to(device); G.apply(_dcgan_init)
    D = DCGANDiscriminator().to(device);       D.apply(_dcgan_init)
    D.fc[-1] = nn.Identity()                  # no Sigmoid for LSGAN

    opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
    fixed_z = torch.randn(64, noise_dim, device=device)
    history = {"G": [], "D": []}

    for epoch in range(1, epochs + 1):
        G.train(); D.train()
        ep_G = ep_D = 0.0
        for x_real, _ in dl:
            B = x_real.size(0); x_real = x_real.to(device)

            z = torch.randn(B, noise_dim, device=device)
            x_fake = G(z).detach()
            loss_D = 0.5 * F.mse_loss(D(x_real), torch.ones(B,1,device=device)) + \
                     0.5 * F.mse_loss(D(x_fake), torch.zeros(B,1,device=device))
            opt_D.zero_grad(); loss_D.backward(); opt_D.step()

            z = torch.randn(B, noise_dim, device=device)
            loss_G = 0.5 * F.mse_loss(D(G(z)), torch.ones(B,1,device=device))
            opt_G.zero_grad(); loss_G.backward(); opt_G.step()

            ep_G += loss_G.item(); ep_D += loss_D.item()

        n = len(dl)
        history["G"].append(ep_G/n); history["D"].append(ep_D/n)
        print(f"[LSGAN] Epoch {epoch:02d}/{epochs} | G {ep_G/n:.4f} | D {ep_D/n:.4f}")

    return G, D, history

lsgan_G, lsgan_D, lsgan_history = train_lsgan(epochs=30)

In [ ]:
variants = {
    "Vanilla GAN": gan_history,
    "DCGAN":       dcgan_history,
    "WGAN-GP":     wgan_history,
    "LSGAN":       lsgan_history,
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (name, h) in zip(axes.flat, variants.items()):
    ax.plot(h["G"], label="G"); ax.plot(h["D"], label="D")
    ax.set_title(name); ax.set_xlabel("Epoch"); ax.legend()
plt.suptitle("GAN Variants — Loss Curves", fontweight="bold", fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
print(f"\n{'Model':<16} {'G Params':>12} {'Final G Loss':>14}")
print("─" * 44)
summary = [
    ("VAE",       vae,     {"G": vae_history["loss"]}),
    ("GAN",       gan_G,   gan_history),
    ("CGAN",      cgan_G,  cgan_history),
    ("DCGAN",     dcgan_G, dcgan_history),
    ("DCGAN+SN",  dcgan_sn_G, dcgan_sn_history),
    ("WGAN-GP",   wgan_G,  wgan_history),
    ("LSGAN",     lsgan_G, lsgan_history),
]
for name, mdl, h in summary:
    p = sum(x.numel() for x in mdl.parameters())
    key = "loss" if "loss" in h else "G"
    print(f"{name:<16} {p:>12,} {h[key][-1]:>14.4f}")

print(f"\nPS6 Augmentation: Baseline={base_acc:.4f} | Augmented={aug_acc:.4f} | Δ={aug_acc-base_acc:+.4f}")